# Names Are Not Boxes

This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Names Are Not Boxes** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


## Why this topic becomes easier once you watch the runtime carefully

When people struggle with **Names Are Not Boxes**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


## Questions worth answering before we go any further

- Replace the box model with the reference model
- Understand aliasing clearly
- See why assignment does not imply copying
- Prepare for function arguments and mutation rules


## What changes in memory while this code runs

When you bind two names to the same list, both names carry references to one shared list object. That one object lives in memory once, but several names may lead to it. That is why one change can appear through multiple names.


In [1]:
items = [1, 2, 3]
alias = items
print(id(items), id(alias))
alias.append(4)
print(items)
print(alias)


2437221032640 2437221032640
[1, 2, 3, 4]
[1, 2, 3, 4]


## What compiled bytecode reveals about this code

Even assignment bytecode is revealing. Python evaluates the right side, then stores the resulting object reference under the target name. It is a store operation, not an implicit clone operation.


In [2]:
import dis

def bind():
    a = [1, 2]
    b = a
    return a, b

dis.dis(bind)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (1)
              4 LOAD_CONST               2 (2)
              6 BUILD_LIST               2
              8 STORE_FAST               0 (a)

  5          10 LOAD_FAST                0 (a)
             12 STORE_FAST               1 (b)

  6          14 LOAD_FAST                0 (a)
             16 LOAD_FAST                1 (b)
             18 BUILD_TUPLE              2
             20 RETURN_VALUE


## A slower walk through the main ideas

### Assignment binds a name
The central event is binding, not copying.

### Aliasing is normal
Two or more names can point to the same object and that is not a bug by itself.

### Mutation changes the object, not the binding
If an object changes in place, every reference to that object observes the change.

### Rebinding changes the name, not the old object
When a name is rebound, the old object may continue to exist if something else still refers to it.


## Aliasing with a mutable object

The list is shared, so a mutation through one name is visible through the other.


In [3]:
left = [10, 20]
right = left
right.append(30)
print(left)
print(right)


[10, 20, 30]
[10, 20, 30]


## Rebinding versus mutation

Here `left` is moved to a new list, so `right` still refers to the earlier one.


In [4]:
left = [10, 20]
right = left
left = left + [30]
print(left)
print(right)


[10, 20, 30]
[10, 20]


## Function arguments follow the same binding idea

The parameter name inside the function becomes another reference to the passed object.


In [5]:
def add_value(bucket):
    bucket.append(99)

nums = [1, 2]
add_value(nums)
print(nums)


[1, 2, 99]


## Where this usually shows up in real code

This is not only a classroom topic. **Names Are Not Boxes** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


## Common traps that still catch careful people

- Saying “Python passed it by reference” without carefully explaining what that means
- Expecting `a = b` to make a full copy
- Confusing rebinding with mutation


## Questions that reveal whether this idea is really clear yet

- What is aliasing?
- Why does changing a list inside a function affect the caller sometimes?
- What really happens during assignment?


## What I would really want you to remember later

- Names are references, not boxes.
- Aliasing is normal.
- Mutation and rebinding are different events.


## One last grounding thought

If this notebook did its job, **Names Are Not Boxes** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
